# Semantic View Eval CI/CD Gate

**Programmatic regression gate for Cortex Analyst semantic views.**

This notebook is the *gate*, not the *runner*. It reads two evaluation runs that
have already completed and decides whether the change is safe to promote:

1. Read the baseline run (scored against the frozen `FULFILLMENT_SV_V1`)
2. Inspect failures — which ambiguity traps caught the model
3. Confirm the optimized view under test carries the SAME question set
4. Read the optimized run and assert it scored **above the baseline** (the judge is
   non-deterministic, so the gate asserts ordering, not a fixed decimal)
5. Persist the comparison as an audit trail

**Why it reads runs instead of starting them — a real platform constraint.**
`EXECUTE_AI_EVALUATION('START')` cannot be driven from a notebook. The run
registers, its status stays `CREATED` forever, and **no error is raised** — the
notebook would simply hang. From a stored procedure the same call fails loudly
and names the mechanism: *"Query called from a stored procedure contains a
function with side effects `[SYSTEM$CORTEX_ANALYST_CREATE_ANALYST_EVAL_OPTIMIZATION]`"*.
Verified across five notebook runs on 2026-08-14, including as ACCOUNTADMIN, so
this is not a privilege problem. The identical call from a SQL worksheet or the
Snowflake CLI completes in ~3.5 minutes.

So in a real pipeline the split is: **your CI step starts the evals** (worksheet,
CLI, or task), **this notebook gates on the results**. That separation is good
practice anyway — the gate stays fast, deterministic, and re-runnable.

**Requirements:** SYSADMIN role, `AGENT_EVAL_DEMO_WH` warehouse, `PYPI_ACCESS_INTEGRATION` EAI.

In [ ]:
-- CRITICAL: Eval resolves objects in the session schema.
-- Must match where the semantic view lives.
USE ROLE SYSADMIN;
USE DATABASE AGENT_EVAL_DEMO;
USE SCHEMA AI;
USE WAREHOUSE AGENT_EVAL_DEMO_WH;

## Configuration

Define run parameters. In CI/CD, these come from environment variables or pipeline config.

In [ ]:
import json

# -- CI/CD Parameters --
DATABASE = 'AGENT_EVAL_DEMO'
SCHEMA = 'AI'

# TWO views. This is the whole point of the gate: the baseline must be scored
# on the FROZEN v1 view, because Step G replaced FULFILLMENT_SV in place with
# the optimized v2 definition. Pointing the baseline at FULFILLMENT_SV scores
# v2 twice, lift comes out ~0.00, and the gate FAILS for a reason that has
# nothing to do with the change under test.
BASELINE_VIEW  = 'FULFILLMENT_SV_V1'   # frozen weak v1
OPTIMIZED_VIEW = 'FULFILLMENT_SV'      # current (v2 / the "PR")

# This notebook READS two evaluation runs that have ALREADY COMPLETED; it does
# not start them. That is a hard platform constraint, not a style choice:
# EXECUTE_AI_EVALUATION('START') cannot be driven from a notebook. Verified
# 2026-08-14 -- the run registers, its status stays CREATED forever, and NO
# error is raised, so the notebook simply hangs until its poll ceiling. The
# same call from a stored procedure fails loudly and names the mechanism:
#   "contains a function with side effects
#    [SYSTEM$CORTEX_ANALYST_CREATE_ANALYST_EVAL_OPTIMIZATION]"
# Role is NOT the discriminator -- ACCOUNTADMIN stalls from a notebook too,
# while the identical call from a SQL worksheet completes in ~3.5 min.
#
# So the runs are started by SQL (sql/05_eval_baseline.sql, sql/06_semantic_v2.sql)
# and this notebook is the ASSERT step -- exactly what a CI job does after its
# test stage reports. It runs in seconds and cannot stall live.
BASELINE_RUN  = 'BASELINE_V1_FINAL'
OPTIMIZED_RUN = 'OPTIMIZED_V2_FINAL'

# CI/CD gate thresholds
MIN_SCORE = 0.30           # Minimum acceptable avg score
MAX_REGRESSIONS = 2        # Max queries that can regress
MIN_IMPROVEMENT = 0.05     # Minimum absolute score lift required

print(f"Baseline : {DATABASE}.{SCHEMA}.{BASELINE_VIEW}  run={BASELINE_RUN}")
print(f"Optimized: {DATABASE}.{SCHEMA}.{OPTIMIZED_VIEW}  run={OPTIMIZED_RUN}")
print(f"Gate: min_score={MIN_SCORE}, max_regressions={MAX_REGRESSIONS}, min_improvement={MIN_IMPROVEMENT}")


## Step 1: Read the Baseline Evaluation

Confirm the baseline run exists and is fully scored before gating on it.

The baseline is scored against the **frozen** `FULFILLMENT_SV_V1`, not the live
`FULFILLMENT_SV` — the live view has since been replaced in place with the
optimized v2 definition, so re-pointing the baseline at it would score v2 and
report a ~0.65 "baseline". Targeting the frozen view keeps the baseline
reproducible no matter what order things ran in.

The eval engine temporarily removes each verified query from the view while
testing it, so scores reflect genuine generation — not lookup.

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

def scored_row_count(run_name: str, view: str) -> int:
    """How many SCORED questions this run has on this view.

    Column note: GET_ANALYST_AI_EVALUATION_DATA exposes METRIC_STATUS, NOT
    STATUS -- selecting STATUS raises 000904 invalid identifier. The run-level
    STATUS column belongs to EXECUTE_AI_EVALUATION, a different function.
    Filter EVAL_AGG_SCORE IS NOT NULL: a run can have rows present while the
    judge has not scored them yet, and those would drag the mean toward zero.
    """
    rows = session.sql(f"""
        SELECT COUNT(*) AS n
        FROM TABLE(SNOWFLAKE.LOCAL.GET_ANALYST_AI_EVALUATION_DATA(
            '{DATABASE}', '{SCHEMA}', '{view}', 'SEMANTIC VIEW', '{run_name}'
        ))
        WHERE METRIC_NAME = 'sql_correctness'
          AND EVAL_AGG_SCORE IS NOT NULL
    """).collect()
    return int(rows[0]['N']) if rows else 0

for run, view in ((BASELINE_RUN, BASELINE_VIEW), (OPTIMIZED_RUN, OPTIMIZED_VIEW)):
    n = scored_row_count(run, view)
    print(f"  {run:20} on {view:20} -> {n} scored questions")
    assert n > 0, (
        f"Run {run} has no scored rows on {view}. Start it from SQL first "
        f"(sql/05_eval_baseline.sql / sql/06_semantic_v2.sql) -- a notebook "
        f"cannot start an evaluation."
    )


In [ ]:
# (Baseline is already COMPLETED -- verified above. Nothing to poll.)


## Step 2: Inspect Baseline Results

Use `GET_ANALYST_AI_EVALUATION_DATA` (not the NORMALIZED UDTF, which double-counts eval rows).
This returns exactly one row per question per metric — the authoritative source for scores.

In [ ]:
import pandas as pd

def get_eval_results(run_name: str, view: str) -> pd.DataFrame:
    """Retrieve per-question eval scores for a run on a specific view.

    Uses GET_ANALYST_AI_EVALUATION_DATA, which returns 1 row per question.
    Do NOT use GET_AI_OBSERVABILITY_EVENTS_NORMALIZED here -- it double-counts
    eval metric rows (one NULL-score row paired with one scored row).
    """
    # DEDUP IS REQUIRED, not defensive polish. A run can carry a DUPLICATE metric
    # row for one question -- observed 2026-08-17: OPTIMIZED_V2_FINAL returned 21
    # rows for 20 distinct INPUT_IDs after a DELETE + START cycle. Undeduped that
    # (a) skews the mean (0.6667 raw vs 0.700 deduped) and (b) trips the
    # question-set assertion below, because baseline has 20 rows and optimized 21.
    df = session.sql(f"""
        SELECT * FROM TABLE(SNOWFLAKE.LOCAL.GET_ANALYST_AI_EVALUATION_DATA(
            '{DATABASE}', '{SCHEMA}', '{view}', 'SEMANTIC VIEW', '{run_name}'
        ))
        WHERE METRIC_NAME IS NOT NULL
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY INPUT_ID, METRIC_NAME
            ORDER BY EVAL_AGG_SCORE DESC NULLS LAST
        ) = 1
    """).to_pandas()
    return df

baseline_df = get_eval_results(BASELINE_RUN, BASELINE_VIEW)
baseline_score = baseline_df['EVAL_AGG_SCORE'].mean()

print(f"{'='*60}")
print(f"BASELINE RESULTS: {BASELINE_RUN}  (view: {BASELINE_VIEW})")
print(f"{'='*60}")
print(f"  Questions evaluated: {len(baseline_df)}")
print(f"  Avg sql_correctness: {baseline_score:.4f}")
print(f"  Correct (1.0):       {(baseline_df['EVAL_AGG_SCORE'] == 1.0).sum()}")
print(f"  Partial (0<x<1):     {((baseline_df['EVAL_AGG_SCORE'] > 0) & (baseline_df['EVAL_AGG_SCORE'] < 1)).sum()}")
print(f"  Wrong (0.0):         {(baseline_df['EVAL_AGG_SCORE'] == 0.0).sum()}")
print(f"{'='*60}")

# Gate check: is baseline above minimum?
assert baseline_score >= MIN_SCORE, (
    f"GATE FAIL: Baseline score {baseline_score:.4f} < minimum {MIN_SCORE}. "
    f"Model may be broken rather than weak -- investigate before optimizing."
)


## Step 3: Analyze Failures

Inspect the judge explanations to understand *why* queries failed.
This is the most actionable output — it tells you exactly which semantic gaps to fix.

In [ ]:
# Show failures with judge explanations
failures = baseline_df[baseline_df['EVAL_AGG_SCORE'] < 1.0].copy()
failures = failures.sort_values('EVAL_AGG_SCORE')

print(f"\n{'='*60}")
print(f"FAILURE ANALYSIS ({len(failures)} questions below 1.0)")
print(f"{'='*60}\n")

for _, row in failures.iterrows():
    question = str(row.get('INPUT', ''))[:80]
    score = row['EVAL_AGG_SCORE']
    # Extract judge explanation from METRIC_CALLS
    metric_calls = row.get('METRIC_CALLS', '[]')
    if isinstance(metric_calls, str):
        try:
            calls = json.loads(metric_calls)
            explanation = calls[0].get('explanation', 'N/A') if calls else 'N/A'
        except (json.JSONDecodeError, IndexError):
            explanation = 'N/A'
    else:
        explanation = 'N/A'
    
    print(f"  Score: {score:.2f} | Q: {question}")
    print(f"         Reason: {str(explanation)[:120]}")
    print()

### Failure Categories

Group failures by root cause to prioritize optimization efforts.
Common categories in supply-chain semantic views:
- **Ambiguity**: Multiple valid interpretations (on-time, fill rate, units)
- **Missing table**: Required table not declared at all (e.g., cost queries)
- **Missing dimension**: Business names unmapped to technical IDs
- **Mis-scoped**: Question belongs on a different semantic view

In [ ]:
-- Persisted baseline for comparison after optimization.
-- Notebook-scoped table name so it cannot clobber the canonical demo snapshots
-- (EVAL.BASELINE_V1_FINAL_RESULTS / EVAL.OPTIMIZED_V2_FINAL_RESULTS).
CREATE OR REPLACE TABLE EVAL.NB_BASELINE_SNAPSHOT AS
SELECT * FROM TABLE(SNOWFLAKE.LOCAL.GET_ANALYST_AI_EVALUATION_DATA(
    'AGENT_EVAL_DEMO', 'AI', 'FULFILLMENT_SV_V1', 'SEMANTIC VIEW', '{{BASELINE_RUN}}'
))
WHERE METRIC_NAME IS NOT NULL
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY INPUT_ID, METRIC_NAME ORDER BY EVAL_AGG_SCORE DESC NULLS LAST
) = 1;


## Step 4: Confirm the Optimized View Under Test

In CI/CD this step applies the semantic view DDL from the pull request. Here the
"PR" is already deployed as `FULFILLMENT_SV` (v2), so this cell **inspects**
rather than mutates — and checks the one thing that makes the comparison honest:
both views must carry the SAME 20 verified queries.

What changed between v1 and v2:
1. **Clear column descriptions** — not just "Date" or "Quantity"
2. **Custom instructions** — on-time definition, fill rate formulas, fiscal calendar
3. **Added the cost table** — `ZONE_RATE_CARDS`, plus verified queries that teach the composite rate-card join (no FK relationship can express its effective-date range)
4. **Mapping dimensions** — `TENANT_ROLE_MAPPING` so business names resolve to tenant IDs
5. **Declared metrics** — order fill rate, unit fill rate, on-time rate

If the two question sets ever diverge, the measured lift is meaningless. That is
the first thing a data scientist in the room will suspect, so it is asserted here
in code rather than asserted verbally.

In [ ]:
-- In CI/CD this cell runs the semantic view DDL from the PR under test.
-- In this demo the "PR" is already deployed as FULFILLMENT_SV (v2), while the
-- baseline above was scored against the frozen FULFILLMENT_SV_V1.
--
-- Both views carry the SAME 20 verified queries -- that is what makes the
-- before/after comparison honest. If they ever diverge, the lift is meaningless.

DESCRIBE SEMANTIC VIEW AGENT_EVAL_DEMO.AI.FULFILLMENT_SV;


## Step 5: Read the Optimized Evaluation

Same 20 verified queries, same judge — only the semantic view changed.

Like the baseline, this run has already completed; the notebook reads it rather
than starting it (see the platform constraint at the top).

In [ ]:
print(f"Reading optimized eval run: {OPTIMIZED_RUN} on {OPTIMIZED_VIEW}")


In [ ]:
# (Optimized run is already COMPLETED -- verified in Step 1. Nothing to poll.)


In [ ]:
optimized_df = get_eval_results(OPTIMIZED_RUN, OPTIMIZED_VIEW)
optimized_score = optimized_df['EVAL_AGG_SCORE'].mean()

print(f"{'='*60}")
print(f"OPTIMIZED RESULTS: {OPTIMIZED_RUN}  (view: {OPTIMIZED_VIEW})")
print(f"{'='*60}")
print(f"  Questions evaluated: {len(optimized_df)}")
print(f"  Avg sql_correctness: {optimized_score:.4f}")
print(f"  Correct (1.0):       {(optimized_df['EVAL_AGG_SCORE'] == 1.0).sum()}")
print(f"  Partial (0<x<1):     {((optimized_df['EVAL_AGG_SCORE'] > 0) & (optimized_df['EVAL_AGG_SCORE'] < 1)).sum()}")
print(f"  Wrong (0.0):         {(optimized_df['EVAL_AGG_SCORE'] == 0.0).sum()}")
print(f"  Lift vs baseline:    {optimized_score - baseline_score:+.4f}")
print(f"{'='*60}")


## Step 6: CI/CD Gate — Assert No Regression

The gate enforces three conditions:
1. **Score improvement** — optimized must beat baseline by `MIN_IMPROVEMENT`
2. **Regression count** — no more than `MAX_REGRESSIONS` questions can score lower
3. **Minimum floor** — optimized score must exceed `MIN_SCORE`

If any condition fails, the pipeline blocks the merge.

In [ ]:
# Per-question comparison
# Join on INPUT_ID, not the question TEXT: INPUT_ID is the stable per-question
# key, so the comparison cannot silently drop or duplicate a row if wording or
# whitespace differs between runs. Both runs must cover the SAME questions --
# report the shared count so a partial join is visible rather than assumed.
comparison = baseline_df[['INPUT_ID', 'INPUT', 'EVAL_AGG_SCORE']].merge(
    optimized_df[['INPUT_ID', 'EVAL_AGG_SCORE']],
    on='INPUT_ID',
    suffixes=('_baseline', '_optimized')
)
assert len(comparison) == len(baseline_df) == len(optimized_df), (
    f"Question sets differ: baseline={len(baseline_df)}, "
    f"optimized={len(optimized_df)}, shared={len(comparison)}. "
    f"A lift computed across different question sets is meaningless."
)
comparison['delta'] = comparison['EVAL_AGG_SCORE_optimized'] - comparison['EVAL_AGG_SCORE_baseline']

improved = comparison[comparison['delta'] > 0]
regressed = comparison[comparison['delta'] < 0]
unchanged = comparison[comparison['delta'] == 0]

print(f"{'='*60}")
print(f"COMPARISON: {BASELINE_RUN} -> {OPTIMIZED_RUN}")
print(f"{'='*60}")
print(f"  Baseline avg:    {baseline_score:.4f}")
print(f"  Optimized avg:   {optimized_score:.4f}")
print(f"  Absolute lift:   {optimized_score - baseline_score:+.4f}")
print(f"  Relative lift:   {((optimized_score - baseline_score) / baseline_score) * 100:+.1f}%")
print(f"  Shared questions:{len(comparison)}")
print(f"  Improved:        {len(improved)} questions")
print(f"  Regressed:       {len(regressed)} questions")
print(f"  Unchanged:       {len(unchanged)} questions")
print(f"{'='*60}")

if len(regressed) > 0:
    print(f"\nREGRESSED QUERIES:")
    for _, row in regressed.iterrows():
        q = str(row['INPUT'])[:70]
        print(f"  {row['EVAL_AGG_SCORE_baseline']:.2f} -> {row['EVAL_AGG_SCORE_optimized']:.2f} | {q}")

In [ ]:
# ============================================================
# CI/CD GATE DECISION
# ============================================================

gate_results = []
lift = optimized_score - baseline_score

# Check 1: Minimum score floor
check1_pass = optimized_score >= MIN_SCORE
gate_results.append(('Min score floor', check1_pass,
    f"{optimized_score:.4f} >= {MIN_SCORE}"))

# Check 2: Score improvement
check2_pass = lift >= MIN_IMPROVEMENT
gate_results.append(('Score improvement', check2_pass,
    f"lift {lift:+.4f} >= {MIN_IMPROVEMENT}"))

# Check 3: Regression limit
check3_pass = len(regressed) <= MAX_REGRESSIONS
gate_results.append(('Regression limit', check3_pass,
    f"{len(regressed)} regressions <= {MAX_REGRESSIONS} max"))

# Report
print(f"\n{'='*60}")
print(f"CI/CD GATE REPORT")
print(f"{'='*60}")
all_pass = True
for name, passed, detail in gate_results:
    status = 'PASS' if passed else 'FAIL'
    icon = '+' if passed else 'X'
    print(f"  [{icon}] {name}: {status} ({detail})")
    if not passed:
        all_pass = False

print(f"{'='*60}")
if all_pass:
    print(f"  GATE: PASSED — safe to promote semantic view changes.")
else:
    print(f"  GATE: BLOCKED — do not merge. Fix regressions first.")
print(f"{'='*60}")

# In CI/CD, this assertion fails the pipeline
assert all_pass, "CI/CD gate failed — see report above for details."

## Step 7: Persist Results for Audit Trail

Store the comparison in a table for historical tracking.
In production, this feeds a dashboard showing eval score over time.

In [ ]:
-- Snapshot optimized results (notebook-scoped names, see cell above)
CREATE OR REPLACE TABLE EVAL.NB_OPTIMIZED_SNAPSHOT AS
SELECT * FROM TABLE(SNOWFLAKE.LOCAL.GET_ANALYST_AI_EVALUATION_DATA(
    'AGENT_EVAL_DEMO', 'AI', 'FULFILLMENT_SV', 'SEMANTIC VIEW', '{{OPTIMIZED_RUN}}'
))
WHERE METRIC_NAME IS NOT NULL
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY INPUT_ID, METRIC_NAME ORDER BY EVAL_AGG_SCORE DESC NULLS LAST
) = 1;

-- Comparison view for observability
CREATE OR REPLACE VIEW EVAL.NB_EVAL_COMPARISON AS
SELECT
    b.INPUT::VARCHAR AS question,
    b.EVAL_AGG_SCORE AS baseline_score,
    o.EVAL_AGG_SCORE AS optimized_score,
    o.EVAL_AGG_SCORE - b.EVAL_AGG_SCORE AS delta,
    CASE
        WHEN o.EVAL_AGG_SCORE > b.EVAL_AGG_SCORE THEN 'IMPROVED'
        WHEN o.EVAL_AGG_SCORE < b.EVAL_AGG_SCORE THEN 'REGRESSED'
        ELSE 'UNCHANGED'
    END AS status
FROM EVAL.NB_BASELINE_SNAPSHOT b
JOIN EVAL.NB_OPTIMIZED_SNAPSHOT o
  ON b.INPUT::VARCHAR = o.INPUT::VARCHAR;


In [ ]:
-- Final comparison table for the demo
SELECT
    status,
    COUNT(*) AS count,
    AVG(baseline_score) AS avg_baseline,
    AVG(optimized_score) AS avg_optimized,
    AVG(delta) AS avg_improvement
FROM EVAL.NB_EVAL_COMPARISON
GROUP BY status
ORDER BY status;


## Cleanup

Nothing to clean up. This notebook starts no evaluation runs, so it creates no
run names to release.

The canonical runs `BASELINE_V1_FINAL` and `OPTIMIZED_V2_FINAL` are demo assets
and must **not** be deleted — `scripts/doctor.py` asserts both exist before you
present. In a real pipeline you would instead use unique per-build run names
(`BUILD_<git_sha>`), which sidesteps cleanup entirely.

In [ ]:
-- Nothing to clean up: this notebook starts no evaluation runs, so it creates
-- no run names to delete. The canonical runs BASELINE_V1_FINAL and
-- OPTIMIZED_V2_FINAL are demo assets and must NOT be deleted -- doctor.py
-- asserts both exist before you present.
SELECT 'no notebook-created eval runs to clean up' AS cleanup_status;


## Integration Patterns

### As a Snowflake Task (scheduled nightly)
```sql
CREATE OR REPLACE TASK EVAL.NIGHTLY_EVAL_GATE
  WAREHOUSE = AGENT_EVAL_DEMO_WH
  SCHEDULE = 'USING CRON 0 2 * * * America/New_York'
AS
  EXECUTE NOTEBOOK AGENT_EVAL_DEMO.AI.EVAL_CICD_GATING;
```

### As a GitHub Actions step
```yaml
- name: Run eval gate
  run: |
    snow notebook execute AGENT_EVAL_DEMO.AI.EVAL_CICD_GATING \
      --connection ${{ secrets.SNOWFLAKE_CONNECTION }}
```

### Key decisions for your CI/CD pipeline
- **Run names**: Use `BUILD_<git_sha>` for traceability
- **Thresholds**: Start conservative (allow regressions), tighten over time
- **Cost**: Each run bills ~0.01 credits warehouse + AI_COMPLETE judge tokens
- **Duration**: this notebook runs in ~90 seconds (it reads two completed runs; it does not start them). Starting an eval from SQL costs ~4-5 minutes per run over 20 verified queries.